# Fun Visualisations of the Tracked Events using `plotX`

The events tracked on the native ICON mesh in `02_id_track_events.ipynb`. Frames are drawn by interpolating to a 0.1° grid with a pre-computed KD-tree, as described under [Unstructured Grid Usage](https://marex.readthedocs.io/en/latest/guide/visualisation.html#unstructured-meshes).

In [ ]:
from getpass import getuser
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from IPython.display import Video

import marEx
import marEx.helper as hpc

In [ ]:
scratch_dir = Path("/scratch") / getuser()[0] / getuser()  # Lustre scratch
out_dir = scratch_dir / "marEx" / "examples" / "unstructured"  # where notebooks 01 and 02 wrote their output
plot_dir = out_dir / "plots"
# Nearest-neighbour indices from the mesh to a 0.1° lat/lon grid, for drawing frames
fpath_ckdtree = scratch_dir / "marEx" / "examples" / "grid_files" / "ckdtree" / "rectgrids"

movie_period = slice("2013-01-01", "2013-12-31")
example_days = slice("2013-01-01", "2013-01-06")
example_year = slice("2013-01-01", "2013-12-31")

# Movie frames render one per task without threads: 16 single-threaded workers x 7 GB
n_workers, threads_per_worker, memory_limit = 16, 1, "7GB"

In [ ]:
plot_dir.mkdir(parents=True, exist_ok=True)

# memory_limit is passed explicitly because on an HPC node Dask sizes workers from the whole node's memory,
# not from the job's allocation, and without it the workers can claim more than the job was granted.
client = hpc.start_local_cluster(
    n_workers=n_workers,
    threads_per_worker=threads_per_worker,
    memory_limit=memory_limit,
    scratch_dir=scratch_dir / "marEx" / "dask",
)

# Records wall time, peak memory, spill and worker restarts for each `monitor.stage(...)` block below
monitor = hpc.ResourceMonitor(client)

In [ ]:
marEx.specify_grid(grid_type="unstructured", fpath_ckdtree=fpath_ckdtree)

In [ ]:
extreme_events_ds = xr.open_zarr(out_dir / "events_merged.zarr", chunks={})
extreme_events_ds

In [ ]:
# Extract the Events ID Field
ID_field = extreme_events_ds.ID_field
ID_field

## Make a Movie using `plotX.animate`

`animate()` renders one frame per task and cannot use threads, which is why this notebook's cluster runs one thread per
worker. Each movie is written to `plot_dir`, and the copy shown here is the same file, kept in the repository's
`media/` folder. See [Animation](https://marex.readthedocs.io/en/latest/guide/visualisation.html#animations).

In [ ]:
with monitor.stage("Movie: event IDs"):
    ID_field_subset = ID_field.sel(time=movie_period)

    # Setup Plotting Configuration & Animate
    config = marEx.PlotConfig(plot_IDs=True, ckdtree_res=0.1)
    ID_field_subset.plotX.animate(config, plot_dir=plot_dir, file_name="movie_ID_field_unstruct")

In [ ]:
Video("../../media/movie_ID_field_unstruct.mp4", width=640, height=480, embed=False)

In [ ]:
# The anomaly is in the detection output, not the tracked-events store
extremes_ds = xr.open_zarr(out_dir / "extremes_shifting_hobday.zarr", chunks={})

with monitor.stage("Movie: anomaly"):
    anomaly_subset = extremes_ds.dat_anomaly.sel(time=movie_period)
    anomaly_hot = anomaly_subset.where(anomaly_subset > 0, 0)

    # Setup Plotting Configuration & Animate
    config = marEx.PlotConfig(var_units="Temperature Anomaly", cmap="inferno", cperc=[0, 98], ckdtree_res=0.1)
    movie_anomaly = anomaly_hot.plotX.animate(config, plot_dir=plot_dir, file_name="movie_anomaly_unstruct")

In [ ]:
Video("../../media/movie_anomaly_unstruct.mp4", width=640, height=480, embed=False)

The same year with centroid-based partitioning (`nn_partitioning=False`):

In [ ]:
with monitor.stage("Movie: centroid partitioning IDs"):
    ID_field_centroid_subset = xr.open_zarr(out_dir / "events_merged_centroid.zarr", chunks={}).ID_field.sel(time=movie_period)

    # Setup Plotting Configuration & Animate
    config = marEx.PlotConfig(plot_IDs=True, ckdtree_res=0.1)
    ID_field_centroid_subset.plotX.animate(config, plot_dir=plot_dir, file_name="movie_ID_field_centroid_unstruct")

In [ ]:
Video("../../media/movie_ID_field_centroid_unstruct.mp4", width=640, height=480, embed=False)

## Plot Consecutive Days of Extreme Events

In [ ]:
# Choose a subset of the ID field
ID_field_subset = ID_field.sel(time=example_days)

# Setup Plotting Configuration
config = marEx.PlotConfig(plot_IDs=True)
fig, ax = ID_field_subset.plotX.multi_plot(config, col="time", col_wrap=3);

## Plot Consecutive Months of Extreme Events

In [ ]:
# The first day of each month, selected directly: resample(time="MS").first() is rejected by flox when each
# month spans many single-day chunks, and on a daily series the selection is identical.
ID_field_year = ID_field.sel(time=example_year)
ID_field_subset_day1 = ID_field_year.isel(time=(ID_field_year.time.dt.day == 1))

# Setup Plotting Configuration
config = marEx.PlotConfig(plot_IDs=True)
fig, ax = ID_field_subset_day1.plotX.multi_plot(config, col="time", col_wrap=3);

## Plot Global Extreme Event Frequency

In [ ]:
# Calculate the frequency of Event Occurrence
event_frequency = (ID_field > 0).astype(float).mean("time")

# Setup Plotting Configuration
config = marEx.PlotConfig(var_units="MHW Frequency", cmap="hot_r", cperc=[0, 96], grid_labels=True)
fig, ax, im = event_frequency.plotX.single_plot(config)

## Plot Seasonal Extreme Event Frequency

In [ ]:
# Calculate the frequency of Event Occurrence per season
seasonal_frequency = (ID_field > 0).groupby("time.season").mean(dim="time")

# Setup Plotting Configuration
config = marEx.PlotConfig(var_units="MHW Frequency", cmap="hot_r", cperc=[0, 96], grid_labels=True)
fig, ax = seasonal_frequency.plotX.multi_plot(config, col="season", col_wrap=2);

## Find & Plot a Few of the Longest Events

In [ ]:
events_duration = extreme_events_ds.time_end - extreme_events_ds.time_start
longest_events = events_duration.sortby(events_duration, ascending=False).ID

for ID in longest_events[:10].values:
    print(
        f"ID: {ID:<6}   Start Day: {extreme_events_ds.time_start.sel(ID=ID).dt.strftime('%Y-%m-%d').values}  -->  Duration: {events_duration.sel(ID=ID).dt.days.values:<4} days"
    )

In [ ]:
# Local duration of the longest events, one reduction per event. Comparing the ID field against all nine
# events at once builds a (9, time, ncells) array that does not fit in the cluster on this mesh.
ids = longest_events[:9].values
long_events_local_duration = xr.concat([(ID_field == int(ID)).sum("time") for ID in ids], dim="ID").assign_coords(ID=("ID", ids))

# Setup Plotting Configuration
config = marEx.PlotConfig(var_units="Duration (days)", cmap="hot_r", cperc=[0, 100])
fig, ax = long_events_local_duration.plotX.multi_plot(config, col="ID", col_wrap=3);

## ID Maximum and Mean Event Area

In [ ]:
areas_max = extreme_events_ds.area.max().compute() / (1.0e3**2)
print(f"Maximum object area: {areas_max.values:.0f} km²")

areas_mean = extreme_events_ds.area.mean().compute() / (1.0e3**2)
print(f"Mean object area: {areas_mean.values:.0f} km²")

## Time-Series Plot of Events Area

In [ ]:
area_mean = extreme_events_ds.area.mean("ID").resample(time="ME").mean() / (1.0e3**2)

area_10 = extreme_events_ds.area.reduce(np.nanpercentile, q=10, dim="ID").resample(time="ME").mean() / (1.0e3**2)
area_90 = extreme_events_ds.area.reduce(np.nanpercentile, q=90, dim="ID").resample(time="ME").mean() / (1.0e3**2)

plt.figure(figsize=(15, 6))
area_mean.plot(label="Mean Area", color="k", lw=2)
plt.fill_between(area_mean.time.values, area_10, area_90, alpha=0.5)
plt.ylim([0, 8e7])
plt.ylabel(r"Event Area [km$^2$]");

## Hovmüller Plot of MHW Monthly Frequency vs Latitude
We need to first bin the unstructured data in latitude.

In [ ]:
# Group by latitude bins and calculate mean within each bin
lat_bins = np.linspace(-90.0, 90.0, 360 + 1)
binned_presence = (ID_field > 0).groupby_bins(ID_field.lat, bins=lat_bins).mean(dim="ncells")

# Calculate bin centers for better labelling
bin_centers = [(lat_bins[i] + lat_bins[i + 1]) / 2 for i in range(len(lat_bins) - 1)]
binned_presence = binned_presence.assign_coords(lat_center=("lat_bins", bin_centers))

# Resample in time to get monthly means
spatial_presence = binned_presence.resample(time="ME").mean()

fig, ax = plt.subplots(figsize=(15, 6))
im = spatial_presence.plot(
    ax=ax,
    cmap="hot",
    x="time",
    y="lat_center",
    cbar_kwargs={"label": "MHW Presence Frequency", "extend": "both"},
)

ax.set_xlabel("Time")
ax.set_ylabel("Latitude")
ax.grid(True, linestyle="--", alpha=0.6)

## Seasonal Cycle of MHW vs Latitude
We again need to bin the unstructured data in latitude. (Reuse the `binned_presence` from above.)

In [ ]:
spatial_presence_seasonal = binned_presence.groupby("time.dayofyear").mean()

fig, ax = plt.subplots(figsize=(15, 6))
im = spatial_presence_seasonal.plot(
    ax=ax,
    cmap="hot",
    x="dayofyear",
    y="lat_center",
    cbar_kwargs={"label": "MHW Seasonal Frequency", "extend": "both"},
)

months = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun",
    "Jul",
    "Aug",
    "Sep",
    "Oct",
    "Nov",
    "Dec",
]
days_per_month = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
month_positions = np.cumsum([0] + days_per_month[:-1]) + 15

ax.set_xticks(month_positions)
ax.set_xticklabels(months)
ax.set_xlabel("Month")
ax.set_ylabel("Latitude")
ax.grid(True, linestyle="--", alpha=0.6)

## Wall Time and Memory

One row per stage, measured while the notebook ran on the cluster configured at the top. Peak worker memory is the
summed process memory the workers report to the scheduler, which is the quantity Dask compares against `memory_limit`.
The client (this kernel) is counted separately.

In [ ]:
monitor.summary()

In [ ]:
client.close()